In [114]:
import torch
import einops

In [115]:
from utils.logsumexp import logsumexp_infsafe as logsumexp

In [116]:
def vector_gather(vectors, indices):
    """
    Gathers (batched) vectors according to indices.
    Arguments:
        vectors: Tensor[N, L, D]
        indices: Tensor[N, K] or Tensor[N]
    Returns:
        Tensor[N, K, D] or Tensor[N, D]
    """
    N, L, D = vectors.shape
    squeeze = False
    if indices.ndim == 1:
        squeeze = True
        indices = indices.unsqueeze(-1)
    N2, K = indices.shape
    assert N == N2
    indices = einops.repeat(indices, "N K -> N K D", D=D)
    out = torch.gather(vectors, dim=1, index=indices)
    if squeeze:
        out = out.squeeze(1)
    return out

In [117]:
def dag_loss(targets, transition_matrix, emission_probs):
    batch_size, m = targets.shape
    _, l, vocab_size = emission_probs.shape
    dp = torch.ones((batch_size, m, l))
    dp[dp == 1] = -float('inf')
    initial_probs = torch.gather(emission_probs, dim=2, index=targets[:, 0].unsqueeze(1).unsqueeze(2))
    dp[:, 0, 0] = initial_probs.squeeze(2).squeeze(1)
    # assumes that transition_matrix and emission_probs are already in log space
    # also we need to tranpose emission_probs so it is vocab_size x l
    # so the vector gather works
    emission_probs = emission_probs.transpose(1, 2)
    for i in range(1, m):
        dp[:, i, :] = vector_gather(emission_probs, targets[:, i]) + (logsumexp(dp[:, i-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1))
    return dp

In [118]:
def process_dp(dp, target_lens, vertex_lens):
    """
    Processes the dynamic programming table (dp) to extract the correct loss values.
    The target lengths and vertex lengths are needed to determine which values to extract
    and which values are a result of padding and should be ignored.

    Args:
        dp (torch.Tensor): The dynamic programming table of shape (batch_size, m, l).
        target_lens (torch.Tensor): A tensor of shape (batch_size,) that describes the length of each target sequence.
        vertex_lens (torch.Tensor): A tensor of shape (batch_size,) that describes the number of non-padding vertices for each batch.

    Returns:
        torch.Tensor: The values corresponding to the last target and last vertex of shape (batch_size,).
    """
    dp_values = vector_gather(dp, target_lens - 1)
    values = torch.gather(dp_values, dim=1, index=(vertex_lens - 1).unsqueeze(-1))
    return values

## Test Case 1

In [119]:
transition_matrix1 = torch.tensor(
    [
        [0, 0.8, 0.1, 0.1],
        [0, 0, 0.8, 0.2],
        [0, 0, 0, 1],
        [0, 0, 0, 0]
    ]
)
emission_matrix1 = torch.tensor(
    [
        [0.9, 0.1, 0, 0],
        [0, 0.8, 0.2, 0],
        [0, 0, 0.9, 0.1],
        [0, 0.3, 0, 0.7]
    ]
)
targets = torch.tensor([0, 1, 2, 3])
target_lens = torch.tensor([4])
vertex_lens = torch.tensor([4])
expected_answer1 = torch.tensor(
    [
        transition_matrix1[0][1],
        transition_matrix1[1][2],
        transition_matrix1[2][3],
        emission_matrix1[0][0],
        emission_matrix1[1][1],
        emission_matrix1[2][2],
        emission_matrix1[3][3]
    ]
)
expected_answer1 = torch.prod(expected_answer1)

In [120]:
transition_matrix1 = torch.log(transition_matrix1)
emission_matrix1 = torch.log(emission_matrix1)

In [121]:
acyclic_mask1 = torch.tril(torch.ones((4, 4)))

In [122]:
transition_matrix1 = transition_matrix1.masked_fill(acyclic_mask1 != 0, -float('inf'))

In [123]:
out_1 = dag_loss(targets.unsqueeze(0), transition_matrix1.unsqueeze(0), emission_matrix1.unsqueeze(0))

In [124]:
out_1, torch.exp(out_1), expected_answer1, process_dp(out_1, target_lens, vertex_lens), torch.exp(process_dp(out_1, target_lens, vertex_lens))

(tensor([[[-0.1054,    -inf,    -inf,    -inf],
          [   -inf, -0.5516,    -inf, -3.6119],
          [   -inf,    -inf, -0.8802,    -inf],
          [   -inf,    -inf,    -inf, -1.2368]]]),
 tensor([[[0.9000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.5760, 0.0000, 0.0270],
          [0.0000, 0.0000, 0.4147, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.2903]]]),
 tensor(0.2903),
 tensor([[-1.2368]]),
 tensor([[0.2903]]))

## Test Case 2

In [125]:
transition_matrix2 = torch.tensor(
    [
        [0, 0.8, 0, 0.2],
        [0.6, 0.1, 0.2, 0.1],
        [0, 0, 0, 1],
        [0, 0, 0, 0]
    ]
)
emission_matrix2 = torch.tensor(
    [
        [0.5, 0, 0.5, 0],
        [0, 0.8, 0.2, 0],
        [0, 0, 0.9, 0.1],
        [0, 0.1, 0, 0.9]
    ]
)
targets2 = torch.tensor([0, 1, 2, 3])
target_lens = torch.tensor([4])
vertex_lens = torch.tensor([4])
expected_answer2 = torch.tensor(
    [
        transition_matrix2[0][1],
        transition_matrix2[1][2],
        transition_matrix2[2][3],
        emission_matrix2[0][0],
        emission_matrix2[1][1],
        emission_matrix2[2][2],
        emission_matrix2[3][3]
    ]
)
expected_answer2 = torch.prod(expected_answer2)

In [126]:
transition_matrix2 = torch.log(transition_matrix2)
emission_matrix2 = torch.log(emission_matrix2)

In [127]:
acycle_mask2 = torch.tril(torch.ones((4, 4)))

In [128]:
transition_matrix2 = transition_matrix2.masked_fill(acycle_mask2 != 0, -float('inf'))

In [129]:
out_2 = dag_loss(targets2.unsqueeze(0), transition_matrix2.unsqueeze(0), emission_matrix2.unsqueeze(0))

In [130]:
out_2, torch.exp(out_2), expected_answer2, process_dp(out_2, target_lens, vertex_lens), torch.exp(process_dp(out_2, target_lens, vertex_lens))

(tensor([[[-0.6931,    -inf,    -inf,    -inf],
          [   -inf, -1.1394,    -inf, -4.6052],
          [   -inf,    -inf, -2.8542,    -inf],
          [   -inf,    -inf,    -inf, -2.9596]]]),
 tensor([[[0.5000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.3200, 0.0000, 0.0100],
          [0.0000, 0.0000, 0.0576, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0518]]]),
 tensor(0.0518),
 tensor([[-2.9596]]),
 tensor([[0.0518]]))

## Test case 3

In [131]:
transition_matrix3 = torch.tensor(
    [
        [0, 0.8, 0, 0.1, 0.1],
        [0.7, 0, 0.2, 0, 0.1],
        [0, 0, 0, 0.8, 0.2],
        [0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0]
    ]
)
emission_matrix3 = torch.tensor(
    [
        [0.5, 0, 0.5, 0],
        [0, 0.8, 0.2, 0],
        [0, 0, 0.9, 0.1],
        [0, 0.1, 0, 0.9],
        [0.3, 0, 0.3, 0.4]
    ]
)
targets3 = torch.tensor([0, 1, 2, 3])
target_lens = torch.tensor([4])
vertex_lens = torch.tensor([4])
expected_answer3 = torch.tensor(
    [
        transition_matrix3[0][1],
        transition_matrix3[1][2],
        transition_matrix3[2][3],
        emission_matrix3[0][0],
        emission_matrix3[1][1],
        emission_matrix3[2][2],
        emission_matrix3[3][3]
    ]
)
expected_answer3 = torch.prod(expected_answer3)

In [132]:
transition_matrix3 = torch.log(transition_matrix3)
emission_matrix3 = torch.log(emission_matrix3)

In [133]:
acycle_mask3 = torch.tril(torch.ones((5, 5)))

In [134]:
transition_matrix3 = transition_matrix3.masked_fill(acycle_mask3 != 0, -float('inf'))

In [135]:
out_3 = dag_loss(targets3.unsqueeze(0), transition_matrix3.unsqueeze(0), emission_matrix3.unsqueeze(0))

In [136]:
out_3, torch.exp(out_3), expected_answer3, process_dp(out_3, target_lens, vertex_lens), torch.exp(process_dp(out_3, target_lens, vertex_lens))

(tensor([[[-0.6931,    -inf,    -inf,    -inf,    -inf],
          [   -inf, -1.1394,    -inf, -5.2983,    -inf],
          [   -inf,    -inf, -2.8542,    -inf, -4.5008],
          [   -inf,    -inf,    -inf, -3.1827, -5.3800]]]),
 tensor([[[0.5000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.3200, 0.0000, 0.0050, 0.0000],
          [0.0000, 0.0000, 0.0576, 0.0000, 0.0111],
          [0.0000, 0.0000, 0.0000, 0.0415, 0.0046]]]),
 tensor(0.0415),
 tensor([[-3.1827]]),
 tensor([[0.0415]]))